# Suite VIS — Visitors protocol conformance

Every class that plays a visitor role must implement its `Visitors` protocol completely, with matching arity:
Proxies builders, Plain writers, the Reachable collector, the Validators recorder, and the Comparison recorders. Proxies instances implement
`Visitable`, not a visitor protocol.

In [ ]:
import typing

from mbse.Schemas.Framework import Comparison, Plain, Proxies, Reachable, Validators, Visitors
from support import conformance_problems, instance_problems, protocol_methods

store = Proxies.OfStore()

## VIS-01 · The protocols are `typing.Protocol`s and declare what FRAMEWORK.md says

In [ ]:
for name in ["OfAny", "OfNative", "OfProperty", "OfObject", "OfAdjacency", "OfEntry", "OfLink", "OfRelation",
             "OfUnion", "OfIntersection", "OfIndexed", "OfItem", "Visitable"]:
    protocol = getattr(Visitors, name)
    assert typing.Protocol in protocol.__mro__, name
assert set(protocol_methods(Visitors.OfAny)) == {"as_native", "as_object", "as_union", "as_intersection", "as_indexed"}
assert set(protocol_methods(Visitors.OfIndexed)) == {"items", "item", "append", "remove", "clear", "pairs", "at", "put",
                                                    "discard"}
assert set(protocol_methods(Visitors.OfItem)) == {"key", "value"}
assert set(protocol_methods(Visitors.Visitable)) == {"identity", "schema_name", "owner", "accept"}
assert set(protocol_methods(Visitors.OfObject)) == {"properties", "has", "property", "clear", "adjacencies", "adjacency",
                                                   "identify"}
assert "identity" not in protocol_methods(Visitors.OfObject)  # identity belongs to the visited object
assert set(protocol_methods(Visitors.OfLink)) == {"name", "target", "set"}
assert Visitors.Native == int | float | str | bool | bytes

## VIS-02 · Every implementation conforms to its protocol

In [ ]:
V = Visitors
implementations = {
    "Proxies": [(Proxies._ObjectBuilder, V.OfObject), (Proxies._AdjacencySlot, V.OfAdjacency),
                (Proxies._EntryBuilder, V.OfEntry), (Proxies._LinkSlot, V.OfLink),
                (Proxies._PropertySlot, V.OfProperty), (Proxies._AnySlot, V.OfAny), (Proxies._NativeSlot, V.OfNative),
                (Proxies._ObjectData, V.Visitable), (Proxies._RecordBuilder, V.OfObject), (Proxies._RecordBuilder, V.OfUnion),
                (Proxies._RecordBuilder, V.OfIntersection), (Proxies._ListBuilder, V.OfIndexed),
                (Proxies._MapBuilder, V.OfIndexed), (Proxies._ItemSlot, V.OfItem)],
    "Plain": [(Plain._ObjectWriter, V.OfObject), (Plain._AdjacencyWriter, V.OfAdjacency),
              (Plain._EntryWriter, V.OfEntry), (Plain._LinkWriter, V.OfLink), (Plain._PropertyWriter, V.OfProperty),
              (Plain._AnyWriter, V.OfAny), (Plain._NativeWriter, V.OfNative), (Plain._RecordWriter, V.OfUnion),
              (Plain._RecordWriter, V.OfIntersection), (Plain._ListWriter, V.OfIndexed),
              (Plain._KeyedWriter, V.OfIndexed), (Plain._ItemWriter, V.OfItem)],
    "Reachable": [(Reachable._Collector, V.OfObject), (Reachable._Adjacency, V.OfAdjacency),
                  (Reachable._Entry, V.OfEntry), (Reachable._Link, V.OfLink), (Reachable._Ignored, V.OfProperty),
                  (Reachable._Ignored, V.OfAny), (Reachable._Ignored, V.OfNative), (Reachable._Items, V.OfIndexed)],
    "Validators": [(Validators._ObjectRecord, V.OfObject), (Validators._AdjacencyRecord, V.OfAdjacency),
                   (Validators._EntryRecord, V.OfEntry), (Validators._Link, V.OfLink),
                   (Validators._Value, V.OfProperty), (Validators._Value, V.OfAny), (Validators._Value, V.OfNative),
                   (Validators._ObjectRecord, V.OfUnion), (Validators._ObjectRecord, V.OfIntersection),
                   (Validators._ListRecord, V.OfIndexed), (Validators._ItemRecord, V.OfItem)],
    "Comparison": [(Comparison.OfObject, V.OfObject), (Comparison.OfAdjacency, V.OfAdjacency),
                   (Comparison.OfEntry, V.OfEntry), (Comparison.OfLink, V.OfLink),
                   (Comparison.OfProperty, V.OfProperty), (Comparison.OfAny, V.OfAny),
                   (Comparison.OfNative, V.OfNative), (Comparison.OfUnion, V.OfUnion),
                   (Comparison.OfIntersection, V.OfIntersection), (Comparison.OfIndexed, V.OfIndexed),
                   (Comparison.OfItem, V.OfItem)],
}
problems = [p for pairs in implementations.values() for impl, proto in pairs for p in conformance_problems(impl, proto)]
assert problems == [], problems

## VIS-03 · Proxies instances are `Visitable`, not visitors
FRAMEWORK.md: builders and serializers implement `Visitors`; proxies do not.

In [ ]:
for protocol in [V.OfObject, V.OfAdjacency, V.OfEntry]:
    assert conformance_problems(Proxies._ObjectData, protocol), protocol

## VIS-04 · Visitor methods chain: non-query methods return `self`

In [ ]:
from mbse.Schemas.Framework import Schemas as S
from support import text

Rel = S.OfRelation.Builder().name("Rel").links("a", "b").properties(text("k")).create()
Obj = S.OfObject.Builder().name("Obj").ref().properties(text("x")).relations(lambda r: r.name("r").of(Rel).me("a")).create()
store.register(Obj)
store.register(Rel)

builder = store.Obj()
assert builder.property("x", lambda p: None) is builder
assert builder.properties(lambda p: None) is builder
assert builder.clear("x") is builder
assert builder.adjacencies(lambda a: None) is builder
assert builder.adjacency("r", lambda a: None) is builder

slots = []
builder.property("x", slots.append)
prop = slots[0]
assert prop.value(lambda a: None) is prop and prop.clear() is prop
prop.value(slots.append)
any_ = slots[1]
assert any_.as_native(lambda n: None) is any_
any_.as_native(slots.append)
native = slots[2]
assert native.set("v") is native and native.clear() is native

builder.adjacency("r", slots.append)
adjacency = slots[3]
assert adjacency.add(lambda e: slots.append(e)) is adjacency
entry = slots[4]
assert entry.property("k", lambda p: None) is entry and entry.clear("k") is entry
assert entry.links(lambda link: slots.append(link)) is entry
link = slots[5]
assert link.name() == "b" and link.set(store.Obj().create()) is link
assert adjacency.remove(entry) is adjacency and adjacency.entries(lambda e: None) is adjacency

## VIS-05 · `OfRelation` has no implementation yet; an intersection of unions is valid
`OfRelation` is declared but no class implements it. An intersection's parts do not merge, so its parts may be unions.

In [ ]:
from mbse.Schemas.Framework import Schemas as S
from support import raises

Id = S.OfUnion.Builder().branches(lambda b: b.name("i").of(lambda t: t.as_native(int)),
                                  lambda b: b.name("s").of(lambda t: t.as_native(str))).create()
Ids = S.OfIntersection.Builder().parts(lambda p: p.name("a").of(Id), lambda p: p.name("b").of(Id)).create()
assert Ids.validate() == [] and {n: p.type for n, p in Ids.properties.items()} == {"a": Id, "b": Id}

## VIS-06 · Live instances conform too
A class can declare every method yet have an instance attribute shadow one of them: the recorder in `Validators`
once did (its `adjacencies` dict hid its `adjacencies()` method), and VIS-02, which inspects classes, missed it.

In [ ]:
found = []
builder = store.Obj()
found.append((builder, V.OfObject))


def on_property(p):
    found.append((p, V.OfProperty))
    p.value(lambda a: (found.append((a, V.OfAny)), a.as_native(lambda n: found.append((n, V.OfNative)))))


def on_adjacency(a):
    found.append((a, V.OfAdjacency))
    a.add(lambda e: (found.append((e, V.OfEntry)), e.link("b", lambda k: found.append((k, V.OfLink)))))


builder.property("x", on_property)
builder.adjacency("r", on_adjacency)
found.append((store.Obj().create(), V.Visitable))
record = Validators._ObjectRecord()
record.adjacency("r", lambda a: a.add(lambda e: found.append((e, V.OfEntry))))
found += [(record, V.OfObject), (Reachable._Collector(lambda target: None), V.OfObject)]
writer = Plain._ObjectWriter({}, Obj, lambda target: {})
found.append((writer, V.OfObject))
comparison = Comparison.OfObject(Obj)
found.append((comparison, V.OfObject))
comparison.property("x", on_property)
comparison.adjacency("r", on_adjacency)
writer.adjacency("r", lambda a: (found.append((a, V.OfAdjacency)), a.add(lambda e: found.append((e, V.OfEntry)))))
problems = [p for instance, protocol in found for p in instance_problems(instance, protocol)]
assert problems == [], problems
assert record.adjacencies(lambda a: a.name()) is record  # the method the bug hid

## VIS-07 · Every protocol method of every implementation behaves
Serializers, the reachability collector and the validation recorder only receive the calls a `Visitable` makes.
This calls every other protocol method on each of them, and on the parts of the proxy builder that the DSL skips.

In [ ]:
from support import raises

target = store.Obj().create()
seen = []

# Plain writers: every method on objects, properties, values, adjacencies, entries and links.
out = {}
writer = Plain._ObjectWriter(out, Obj, lambda t: {"$ref": "s9", "$schema": t.schema_name()})
assert not writer.has("x")
writer.properties(seen.append)  # nothing is present yet
assert seen == []
writer.property("x", lambda p: p.value(lambda a: a.as_native(lambda n: n.set("v"))))
assert writer.has("x") and out == {"x": "v"}
writer.properties(lambda p: seen.append((p.name(), p.has())))
writer.property("x", lambda p: p.value(lambda a: a.as_native(lambda n: seen.append((n.has(), n.get())))))
writer.property("x", lambda p: p.value(lambda a: a.as_native(lambda n: n.clear())))
assert out == {} and seen == [("x", True), (True, "v")]
writer.property("x", lambda p: p.value(lambda a: a.as_native(lambda n: n.set("w"))).clear())
writer.property("x", lambda p: p.value(lambda a: a.as_native(lambda n: n.set("w"))))
assert writer.clear("x") is writer and out == {}
with raises(KeyError, match="unknown property"):
    writer.property("nope", lambda p: None)


def fill(entry):
    entry.properties(seen.append)  # nothing is present yet
    entry.links(lambda k: (seen.append(k.name()), k.set(target)))
    with raises(NotImplementedError):
        entry.link("b", lambda k: k.target(lambda t: None))
    with raises(KeyError, match="not a link this entry can set"):
        entry.link("a", lambda k: None)  # the object's own link
    entry.property("k", lambda p: p.value(lambda v: v.as_native(lambda n: n.set("K"))))
    seen.append(entry.has("k"))
    entry.properties(lambda p: seen.append(p.name()))


seen.clear()
writer.adjacencies(lambda a: (seen.append((a.name(), a.me())), a.add(fill)))
assert seen == [("r", "a"), "b", True, "k"]
assert out == {"r": [{"b": {"$ref": "s9", "$schema": "Obj"}, "k": "K"}]}
writer.adjacency("r", lambda a: a.entries(lambda e: e.clear("k")))
assert out == {"r": [{"b": {"$ref": "s9", "$schema": "Obj"}}]}
with raises(NotImplementedError):
    writer.adjacency("r", lambda a: a.remove(None))

# The reachability collector ignores native values, collects value objects' entries, and reports only link targets.
found = []
collector = Reachable._Collector(found.append)
assert not collector.has("x") and collector.clear("x") is collector
assert collector.properties(lambda p: found.append(p)) is collector and collector.adjacencies(found.append) is collector
collector.property("x", lambda p: (seen.append(p.name()), seen.append(p.has()), p.clear(),
                                   p.value(lambda a: a.as_native(lambda n: (n.set(1), n.clear())))))
nested = []
for method in ["as_object", "as_union", "as_intersection"]:
    collector.property("x", lambda p, m=method: p.value(lambda a: getattr(a, m)(nested.append)))
assert len(nested) == 3 and all(isinstance(v, Reachable._Collector) and v.identify(None) is v for v in nested)


def collect(entry):
    assert entry.links(found.append) is entry and entry.properties(found.append) is entry
    assert not entry.has("k") and entry.clear("k") is entry
    entry.property("k", lambda p: p.value(lambda a: a.as_native(lambda n: n.set("K"))))
    entry.link("b", lambda k: (seen.append(k.name()), k.set(target)))


collector.adjacency("r", lambda a: (seen.append(a.name()), a.entries(found.append), a.add(collect), a.remove(None)))
assert found == [target]  # only the link target was reported

# The validation recorder keeps everything it is given, and can be read back through the same protocols.
record = Validators._ObjectRecord()
record.property("x", lambda p: p.value(lambda a: a.as_native(lambda n: n.set("v"))))
seen.clear()
record.properties(lambda p: seen.append((p.name(), p.has())))
record.property("x", lambda p: p.value(lambda a: a.as_native(lambda n: seen.append(n.get()))))
assert record.has("x") and seen == [("x", True), "v"]
record.property("x", lambda p: p.clear())
assert not record.has("x")
record.property("x", lambda p: p.value(lambda a: a.as_native(lambda n: n.set("v").clear())))
assert record.clear("x") is record and not record.has("x")


def record_entry(entry):
    entry.link("b", lambda k: k.set(target))
    entry.property("k", lambda p: p.value(lambda a: a.as_native(lambda n: n.set("K"))))


record.adjacency("r", lambda a: a.add(record_entry))
seen.clear()


def read_back(entry):
    entry.links(lambda k: (seen.append(k.name()), k.target(seen.append)))
    entry.properties(lambda p: seen.append(p.name()))
    seen.append(entry.has("k"))
    entry.clear("k")
    seen.append(entry.has("k"))


record.adjacencies(lambda a: (seen.append(a.name()), a.entries(read_back)))
assert seen == ["r", "b", target, "k", True, False]
with raises(NotImplementedError):
    record.adjacency("r", lambda a: a.me())
record.adjacency("r", lambda a: a.entries(lambda e: a.remove(e)))
record.adjacencies(lambda a: a.entries(lambda e: seen.append("left")))
assert "left" not in seen

# Proxy builders: the entry and slot methods the DSL does not use.
seen.clear()


def proxy_entry(entry):
    entry.links(lambda k: (seen.append(k.name()), k.set(target), k.target(seen.append)))
    entry.property("k", lambda p: p.value(lambda a: a.as_native(lambda n: n.set("K"))))
    entry.properties(lambda p: seen.append((p.name(), p.has())))
    seen.append(entry.clear("k").has("k"))
    with raises(AttributeError, match="not a property of this relation"):
        entry.property("nope", lambda p: None)
    with raises(AttributeError):
        entry._private
    with raises(AttributeError):
        entry.nope
    assert getattr(entry, "then", None) is None


builder = store.Obj()
builder.adjacency("r", lambda a: (a.entries(seen.append), a.remove(None)))  # no entries yet
assert getattr(builder, "then", None) is None  # not a schema name
builder.adjacency("r", lambda a: a.add(proxy_entry))
assert seen == ["b", target, ("k", True), False]
with raises(AttributeError, match="'x' is not set"):
    builder.property("x", lambda p: p.value(lambda a: a.as_native(lambda n: n.get())))
builder.property("x", lambda p: (seen.append(p.has()), p.value(lambda a: a.as_native(lambda n: n.set("v"))), p.clear()))
assert not builder.has("x")